In [1]:
import os
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import PowerTransformer


In [2]:
# rutas base
REPO_ROOT = Path('.').resolve()
RUN_ID = datetime.today().strftime('%Y%m%d')
INPUT_DIR = REPO_ROOT / 'data' / 'intermediate' / '03_normalizacion' / RUN_ID
OUTPUT_DIR = REPO_ROOT / 'data' / 'intermediate' / '04_featuring' / RUN_ID
REPORTS_DIR = REPO_ROOT / 'reports' / '04_featuring' / RUN_ID
METRIC_DIR = REPORTS_DIR / 'metricas_generacion'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
METRIC_DIR.mkdir(parents=True, exist_ok=True)

if not INPUT_DIR.exists():
    candidates = sorted((REPO_ROOT / 'data' / 'intermediate' / '03_normalizacion').glob('*/'))
    if candidates:
        INPUT_DIR = candidates[-1]
        print(f'Usando ultima normalizacion encontrada: {INPUT_DIR}')
    else:
        raise FileNotFoundError('No se encontro normalizacion en data/intermediate/03_normalizacion')

print('Rutas configuradas:')
print(f'INPUT_DIR: {INPUT_DIR}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')
print(f'METRIC_DIR: {METRIC_DIR}')


Rutas configuradas:
INPUT_DIR: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\data\intermediate\03_normalizacion\20260112
OUTPUT_DIR: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\data\intermediate\04_featuring\20260112
METRIC_DIR: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\reports\04_featuring\20260112\metricas_generacion


In [3]:
def add_features(df):
    df_extended = df.copy().astype(np.float32)
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    numeric_cols = [col for col in numeric_cols if df[col].nunique() > 2]

    detalle_columnas = []

    for col in numeric_cols:
        try:
            col_data = df[col].replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)

            # Log, sqrt, squared
            df_extended[f"{col}_log"] = np.log1p(np.abs(col_data))
            detalle_columnas.append((col, f"{col}_log", 'log'))

            df_extended[f"{col}_sqrt"] = np.sqrt(np.abs(col_data))
            detalle_columnas.append((col, f"{col}_sqrt", 'sqrt'))

            df_extended[f"{col}_squared"] = col_data ** 2
            detalle_columnas.append((col, f"{col}_squared", 'squared'))

            # Yeo-Johnson
            pt_yj = PowerTransformer(method='yeo-johnson')
            yj_data = pt_yj.fit_transform(col_data.values.reshape(-1, 1)).flatten()
            df_extended[f"{col}_yeojohnson"] = yj_data
            detalle_columnas.append((col, f"{col}_yeojohnson", 'yeojohnson'))

            # Box-Cox (solo si todos los valores son >0)
            if (col_data > 0).all():
                pt_bc = PowerTransformer(method='box-cox')
                bc_data = pt_bc.fit_transform(col_data.values.reshape(-1, 1)).flatten()
                df_extended[f"{col}_boxcox"] = bc_data
                detalle_columnas.append((col, f"{col}_boxcox", 'boxcox'))

        except Exception as e:
            print(f'Error en columna {col}: {e}')

    return df_extended, pd.DataFrame(detalle_columnas, columns=['original', 'generada', 'tipo'])

# Procesar cada subcarpeta
for norm_name in os.listdir(INPUT_DIR):
    norm_dir = INPUT_DIR / norm_name
    if not norm_dir.is_dir():
        continue  # Salta archivos, solo procesa carpetas
    out_dir = OUTPUT_DIR / norm_name
    out_dir.mkdir(parents=True, exist_ok=True)

    try:
        X_train = pd.read_parquet(norm_dir / f'X_train_{norm_name}.parquet')
        X_test = pd.read_parquet(norm_dir / f'X_test_{norm_name}.parquet')
        y_train = pd.read_parquet(norm_dir / f'y_train_{norm_name}.parquet')
        y_test = pd.read_parquet(norm_dir / f'y_test_{norm_name}.parquet')
    except Exception as e:
        print(f'Error leyendo archivos en {norm_name}: {e}')
        continue

    # Guardar version original
    X_train.to_parquet(out_dir / f'X_train_{norm_name}_ORIGINAL.parquet')
    X_test.to_parquet(out_dir / f'X_test_{norm_name}_ORIGINAL.parquet')
    y_train.to_parquet(out_dir / f'y_train_{norm_name}_ORIGINAL.parquet')
    y_test.to_parquet(out_dir / f'y_test_{norm_name}_ORIGINAL.parquet')

    # Aplicar feature engineering SOLO en train
    X_train_fe, df_detalle = add_features(X_train)

    # Replicar las columnas generadas en test
    X_test_fe = X_test.copy()
    for idx, row in df_detalle.iterrows():
        col = row['generada']
        base_col = row['original']
        tipo = row['tipo']
        try:
            base_data = X_test[base_col].replace([np.inf, -np.inf], np.nan).fillna(0)
            if tipo == 'log':
                X_test_fe[col] = np.log1p(np.abs(base_data))
            elif tipo == 'sqrt':
                X_test_fe[col] = np.sqrt(np.abs(base_data))
            elif tipo == 'squared':
                X_test_fe[col] = base_data ** 2
            elif tipo == 'yeojohnson':
                pt_yj = PowerTransformer(method='yeo-johnson')
                X_test_fe[col] = pt_yj.fit(X_train[[base_col]].replace([np.inf, -np.inf], np.nan).fillna(0)).transform(base_data.values.reshape(-1, 1)).flatten()
            elif tipo == 'boxcox':
                if (base_data > 0).all():
                    pt_bc = PowerTransformer(method='box-cox')
                    X_test_fe[col] = pt_bc.fit(X_train[[base_col]].replace([np.inf, -np.inf], np.nan).fillna(0)).transform(base_data.values.reshape(-1, 1)).flatten()
        except Exception as e:
            print(f'Error generando {col} en test: {e}')

    # Asegurar mismo orden de columnas
    X_test_fe = X_test_fe[X_train_fe.columns]

    # Guardar nuevos datasets
    X_train_fe.to_parquet(out_dir / f'X_train_{norm_name}_FE.parquet')
    X_test_fe.to_parquet(out_dir / f'X_test_{norm_name}_FE.parquet')
    y_train.to_parquet(out_dir / f'y_train_{norm_name}_FE.parquet')
    y_test.to_parquet(out_dir / f'y_test_{norm_name}_FE.parquet')


c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\numpy\_core\_methods.py:208: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(x, axis, dtype, out, keepdims=keepdims, where=where)
C:\Users\Gonzalo\AppData\Local\Temp\ipykernel_25324\3625912474.py:25: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_extended[f"{col}_yeojohnson"] = yj_data
C:\Users\Gonzalo\AppData\Local\Temp\ipykernel_25324\3625912474.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_extended[f"{col}_boxcox"

In [4]:
# Guardar CSV con el detalle de las columnas generadas
detalle_file = METRIC_DIR / f'detalle_columnas_{norm_name}.csv'
df_detalle.to_csv(detalle_file, index=False)

# Metricas en consola
resumen = df_detalle['tipo'].value_counts()
print(f'{norm_name} - Ingenieria aplicada')
print(f'Columnas originales: {X_train.shape[1]}')
print(f'Nuevas columnas generadas: {df_detalle.shape[0]}')
print(f'Total final de columnas: {X_train_fe.shape[1]}')
print('Detalle por tipo de transformacion:')
for tipo, count in resumen.items():
print(f'{tipo}: {count}')    
print(f'CSV guardado en: {detalle_file}')


IndentationError: expected an indented block after 'for' statement on line 12 (1209226591.py, line 13)